# AC-2 Parte 1 — Aula 05
## SAC MóveisDesign — Classificação de Intenção com Fallback

Notebook com a resolução dos 4 exercícios: (1) limpeza e lematização de texto, (2) sentence embeddings com Mean Pooling, (3) classificação com Regressão Logística e fallback, (4) comparativo com KNN.

**Dataset:** gerado a partir de `dataset_sintetico_aula05.py` (32 mensagens, 4 intenções: `trocas_devolucoes`, `logistica_entregas`, `suporte_tecnico`, `vendas_orcamento`).

### Setup — instalação das dependências
Rode esta célula primeiro (só precisa rodar uma vez por sessão). Ela instala o gensim e baixa o modelo de português do Spacy.

In [1]:
import sys
!{sys.executable} -m pip install -q gensim
!{sys.executable} -m spacy download pt_core_news_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')


### Geração do dataset sintético

In [2]:
import pandas as pd

# Dataset Sintético de Mensagens do SAC da MóveisDesign S.A.
data_raw = [
    # trocas_devolucoes
    ("Quero devolver este sofa que chegou com rasgo", "trocas_devolucoes"),
    ("Gostaria de trocar minha mesa veio arranhada", "trocas_devolucoes"),
    ("Como faco para solicitar a devolucao do meu estofado?", "trocas_devolucoes"),
    ("O rack veio com defeito e quero trocar", "trocas_devolucoes"),
    ("Preciso devolver a cadeira de escritorio com defeito", "trocas_devolucoes"),
    ("Quero cancelar a compra e pedir extorno do sofá", "trocas_devolucoes"),
    ("Viu meu painel de tv veio quebrado quero troca", "trocas_devolucoes"),
    ("Gostaria de devolver o armário por defeito", "trocas_devolucoes"),
    
    # logistica_entregas
    ("Qual o status da entrega da minha estante?", "logistica_entregas"),
    ("Onde esta meu pedido de poltrona?", "logistica_entregas"),
    ("Qual o prazo de entrega do sofa que comprei?", "logistica_entregas"),
    ("Meu armario de cozinha ainda nao chegou", "logistica_entregas"),
    ("Quero rastrear o transporte da minha mesa de jantar", "logistica_entregas"),
    ("A entrega do guarda roupa esta atrasada", "logistica_entregas"),
    ("Quando chega minha cadeira presidente?", "logistica_entregas"),
    ("Saber dia que chegam meus moveis", "logistica_entregas"),

    # suporte_tecnico
    ("Como montar o painel de tv da sala?", "suporte_tecnico"),
    ("Nao consigo entender o manual de montagem do rack", "suporte_tecnico"),
    ("Faltaram parafusos no kit do meu guarda roupa", "suporte_tecnico"),
    ("Preciso de ajuda para ajustar a porta do armario", "suporte_tecnico"),
    ("A peca B da mesa nao encaixa na peca C", "suporte_tecnico"),
    ("Como regulo a altura da minha cadeira ergonomica?", "suporte_tecnico"),
    ("Voces enviam montador para a estante?", "suporte_tecnico"),
    ("Manual da cama de casal veio em branco", "suporte_tecnico"),

    # vendas_orcamento
    ("Qual o valor da mesa de jantar 6 lugares?", "vendas_orcamento"),
    ("Gostaria de um orcamento de sofa retratil", "vendas_orcamento"),
    ("Voces tem desconto para pagamento via pix na poltrona?", "vendas_orcamento"),
    ("Quanto custa o frete para o guarda roupa de casal?", "vendas_orcamento"),
    ("Tem promocao de comoda este mes?", "vendas_orcamento"),
    ("Qual o preço do armario de cozinha planejado?", "vendas_orcamento"),
    ("Gostaria de comprar um beliche de madeira", "vendas_orcamento"),
    ("Quais as formas de parcelamento do rack?", "vendas_orcamento")
]

# Criar DataFrame e salvar em arquivo CSV
df = pd.DataFrame(data_raw, columns=["mensagem", "intencao"])
df.to_csv("sac_moveis_ac2.csv", index=False)
print(" Dataset 'sac_moveis_ac2.csv' gerado com sucesso!")


 Dataset 'sac_moveis_ac2.csv' gerado com sucesso!


## Exercício 1 — Esteira de Limpeza Avançada e Lemmatization
**Peso:** 0,05 ponto | **Técnica:** Regex + NLTK/Spacy

In [3]:
import re
import nltk
from nltk.corpus import stopwords
import spacy

nltk.download('stopwords', quiet=True)
stop_words_pt = set(stopwords.words('portuguese'))

# Carregar o modelo morfológico do Spacy em Português
nlp = spacy.load("pt_core_news_sm")

def limpar_e_lemmatizar(texto):
    """
    Função para tratar texto bruto:
    1. Converte para minúsculas
    2. Remove caracteres especiais, pontuações e números via Regex
    3. Remove Stop Words e realiza Lemmatization via Spacy
    """
    # TODO 1: Converter para minúsculas
    texto_limpo = texto.lower()

    # TODO 2: Remover tudo que não for letra ou espaço usando re.sub
    texto_limpo = re.sub(r'[^a-záàâãéèêíïóôõöúçñ\s]', '', texto_limpo)

    # Processamento com Spacy
    doc = nlp(texto_limpo)

    # TODO 3: Extrair o lema (token.lemma_) ignorando stop words e espaços vazios
    tokens_filtrados = [
        token.lemma_ for token in doc
        if token.text not in stop_words_pt and not token.is_space and len(token.text) > 1
    ]

    return " ".join(tokens_filtrados)

# === TESTE DO EXERCÍCIO 1 ===
frase_teste = "Gostaria de saber se vocês estão DEVOLVENDO os valores das mesas compradas!!!"
print("Frase Original:", frase_teste)
print("Frase Limpa & Lemmatizada:", limpar_e_lemmatizar(frase_teste))


Frase Original: Gostaria de saber se vocês estão DEVOLVENDO os valores das mesas compradas!!!
Frase Limpa & Lemmatizada: gostar saber devolver valor meso comprada


## Exercício 2 — Sentence Embeddings com Mean Pooling
**Peso:** 0,05 ponto | **Técnica:** Representação Vetorial Densa

> Observação: o enunciado sugere o uso do FastText, mas — como no exemplo dado em aula — utilizamos o modelo leve `glove-wiki-gigaword-50` (50 dimensões) via `gensim.downloader`, substituto equivalente para fins didáticos.

In [4]:
import numpy as np
import pandas as pd
import gensim.downloader as api

print("Carregando modelo de Embeddings FastText (Gensim)...")
# Utilizando o modelo pré-treinado do Gensim em português ou substituto equivalente
fasttext_model = api.load("glove-wiki-gigaword-50") # Exemplo leve de 50 dimensões para testes em aula

def obter_vetor_frase(frase, model):
    """
    Calcula a média dos vetores das palavras de uma frase (Mean Pooling).
    """
    palavras = frase.split()
    vetores = []

    for palavra in palavras:
        if palavra in model:
            # TODO 1: Adicionar o vetor da palavra na lista de vetores
            vetores.append(model[palavra])

    if len(vetores) == 0:
        # Se nenhuma palavra estiver no vocabulário, retorna vetor de zeros
        return np.zeros(model.vector_size)

    # TODO 2: Retornar a média ao longo do eixo 0 (np.mean)
    vetor_medio = np.mean(vetores, axis=0)
    return vetor_medio

# === TESTE DO EXERCÍCIO 2 ===
df = pd.read_csv("sac_moveis_ac2.csv")

# Aplicamos a limpeza e lematização do Exercício 1 antes de gerar os embeddings
df['mensagem_limpa'] = df['mensagem'].apply(limpar_e_lemmatizar)

X_vetores = np.array([obter_vetor_frase(msg, fasttext_model) for msg in df['mensagem_limpa']])
y = df['intencao']
print("Formato da Matriz de Vetores Densos (Exemplos, Dimensões):", X_vetores.shape)


Carregando modelo de Embeddings FastText (Gensim)...


Formato da Matriz de Vetores Densos (Exemplos, Dimensões): (32, 50)


## Exercício 3 — Regressão Logística com Limiar de Confiança (Fallback)
**Peso:** 0,10 ponto | **Técnica:** Regressão Logística ($w \cdot x + b$) com corte de 50%

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. Carregar dados e matriz de vetores
# (X_vetores e y já foram gerados no Exercício 2, a partir de sac_moveis_ac2.csv)

# Treinar o modelo de Regressão Logística
modelo_regressao = LogisticRegression(max_iter=1000)
modelo_regressao.fit(X_vetores, y)

def classificar_com_fallback_linear(mensagem_usuario, modelo, model_emb, limiar=0.50):
    """
    Classifica a intenção e verifica se a probabilidade atinge o limiar mínimo de confiança.
    """
    # 1. Obter o vetor denso da mensagem do usuário (aplicando a mesma limpeza do Ex. 1)
    msg_limpa = limpar_e_lemmatizar(mensagem_usuario)
    vetor_msg = obter_vetor_frase(msg_limpa, model_emb).reshape(1, -1)

    # TODO 1: Obter as probabilidades para cada classe usando predict_proba
    probabilidades = modelo.predict_proba(vetor_msg)[0]

    # TODO 2: Obter a maior probabilidade e o índice correspondente
    max_prob = np.max(probabilidades)
    idx_classe = np.argmax(probabilidades)
    intencao_prevista = modelo.classes_[idx_classe]

    # TODO 3: Aplicar a regra de Fallback
    if max_prob < limiar:
        return "FALLBACK_HUMANO", max_prob
    else:
        return intencao_prevista, max_prob

# === TESTE DO EXERCÍCIO 3 ===
testes = [
    "Quero saber o valor do frete do sofá",             # Intenção esperada: vendas_orcamento
    "Gostaria de ver receitas de bolo de cenoura"       # Frase fora do domínio -> Deve acionar Fallback
]

for t in testes:
    intencao, conf = classificar_com_fallback_linear(t, modelo_regressao, fasttext_model)
    print(f"Frase: '{t}' | Resultado: {intencao} | Confiança: {conf:.2%}")


Frase: 'Quero saber o valor do frete do sofá' | Resultado: trocas_devolucoes | Confiança: 79.92%
Frase: 'Gostaria de ver receitas de bolo de cenoura' | Resultado: FALLBACK_HUMANO | Confiança: 47.73%


## Exercício 4 — Comparativo com KNN (K=3)
**Peso:** 0,05 ponto | **Técnica:** KNN x Regressão Logística no espaço denso

In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# TODO 1: Instanciar e treinar o KNN com n_neighbors=3
modelo_knn = KNeighborsClassifier(n_neighbors=3)
modelo_knn.fit(X_vetores, y)

# Predições nas duas abordagens
y_pred_linear = modelo_regressao.predict(X_vetores)
y_pred_knn = modelo_knn.predict(X_vetores)

# TODO 2: Calcular a acurácia de cada modelo
acuracia_linear = accuracy_score(y, y_pred_linear)
acuracia_knn = accuracy_score(y, y_pred_knn)

print(f" Acurácia - Regressão Logística (Linear): {acuracia_linear:.2%}")
print(f" Acurácia - KNN (Distância K=3): {acuracia_knn:.2%}")

# Pergunta para reflexão do aluno:
# Qual dos dois modelos lida melhor quando temos frases muito curtas ou distantes no espaço vetorial?


 Acurácia - Regressão Logística (Linear): 100.00%
 Acurácia - KNN (Distância K=3): 50.00%
